**Note:** All the functions illustrated in this notebook can be consulted in the file **functions.py**. Not all the functions that appear in the file have been illustrated here. We just focused on the most important ones, which are crucial to understand the workflow and the core content of the project.

## Spark-parallelized Implementation of Clustering Algorithms

Here we describe the Spark-parallelized implementation of both the classic and the Mini-batch versions of $k$-Means algorithm.

We first implemented the classic version of the algorithm in the `classic_kmeans` function reported below. In this function, we start with the `.takeSample` action to initialize `k` centroids by drawing `k` random points from the dataset. During each step of the following `for` loop, first the centroids are broadcast to the Executors. Then, the map-reduce transformations `.map` and `.reduceByKey` are used to assign each point to the closest cluster(this makes use of the custom function `closest_idx`) and compute the vector sums and point count needed to obtain the new centroids. Finally, these are returned to the Driver with the `.collect()` action, where the new centroids are eventually computed. The old broadcast variable is erased from the Workers' memory using `.destroy()`.

```python
def classic_kmeans(rdd_train, k: int, epochs: int, seed: int):
    """Executes the standard K-Means algorithm using Spark"""

    # Initialize centers randomly from training set
    centers = rdd_train.takeSample(False, k, seed)
    
    for _ in range(epochs):
        #Broadcast
        centers_np = np.array(centers)
        bc_centers = rdd_train.context.broadcast(centers_np)
        
        # Map using the broadcasted variable, returns -> (center_idx, (point,count)) of course count will always be 1
        mapped_points = rdd_train.map(lambda x: (closest_idx(x, bc_centers.value), (x, 1)))
        
        # Reduce by key (center_idx) and gives -> (center_idx, (vectorial_sum, population))
        reduced_points = mapped_points.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
        
        # Update centers and gives them to master -> (center_idc, vectorial_sum/population)
        new_centers_rdd = reduced_points.map(lambda x: (x[0], x[1][0] / x[1][1]))
        new_centers_dict = dict(new_centers_rdd.collect())
        
        # Update centers for next iteration
        centers = [new_centers_dict.get(i, centers[i]) for i in range(k)]
        bc_centers.destroy()
        
    return centers

Then, we implemented the Mini-batch version. This approach employs small batches to optimize the classic algorithm. We started from the version proposed by D. Sculley (https://dl.acm.org/doi/10.1145/1772690.1772862), whose pseudocode is reported below. This introduces a gradient-descent approach, introducing of a learning rate term $\eta$ that is updated point-by-point at each step during the inner `for` loop over the batch.

**Algorithm 1:** Mini-batch $k$-Means.
***
1. **Given:** $k$, mini-batch size $b$, iterations $t$, data set $X$
2. Initialize each $\mathbf{c} \in C$ with an $\mathbf{x}$ picked randomly from $X$
3. $\mathbf{v} \leftarrow 0$
4. **for** $i = 1$ to $t$ **do**
5. &nbsp;&nbsp;&nbsp;&nbsp; $M \leftarrow b$ examples picked randomly from $X$
6. &nbsp;&nbsp;&nbsp;&nbsp; **for** $\mathbf{x} \in M$ **do**
7. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $\mathbf{d}[\mathbf{x}] \leftarrow f(C, \mathbf{x})$ &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; *// Cache the center nearest to $\mathbf{x}$*
8. &nbsp;&nbsp;&nbsp;&nbsp; **end for**
9. &nbsp;&nbsp;&nbsp;&nbsp; **for** $\mathbf{x} \in M$ **do**
10. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $\mathbf{c} \leftarrow \mathbf{d}[\mathbf{x}]$ &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; *// Get cached center for this $\mathbf{x}$*
11. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $\mathbf{v}[\mathbf{c}] \leftarrow \mathbf{v}[\mathbf{c}] + 1$ &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; *// Update per-center counts*
12. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $\eta \leftarrow \frac{1}{\mathbf{v}[\mathbf{c}]}$ &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; *// Get per-center learning rate*
13. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $\mathbf{c} \leftarrow (1 - \eta)\mathbf{c} + \eta\mathbf{x}$ &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; *// Take gradient step*
14. &nbsp;&nbsp;&nbsp;&nbsp; **end for**
15. **end for**
***

We then implemented the Spark-parallelized version of the Mini-batch $k$-Means algorithm. First, the centroids are initialized randomly using the `.takeSample` action, and the historical count array `v` is initialized to zero. The `fraction` parameter, quantifying the percentage of samples per batch, is then computed employing the `.count()` action. During the following `for` loop, the centroids are broadcast to the Executors and a random mini-batch is drawn from the dataset using the `.sample()` transformation. Map-reduce transformations(`.map` and .`reduceByKey`) are then employed to assign the points to the nearest centroid, and to compute the vector sums and point counts for each cluster. These are then returned to the Driver via the `.collect()` action. Finally, the centroids are computed with a gradient step: for each cluster, the historical volume `v` is updated and the learning rate `$eta$` is calculated. The centroid is moved towards the center of the current batch according to this learning rate. Lastly, the broadcast variable is then destroyed to free the memory.

```python
def minibatch_kmeans(rdd_train, k: int, b: int, epochs: int, seed: int):
    """Executes the Mini-Batch K-Means algorithm using Spark"""

    # Initialize centers randomly from training set
    centers_list = rdd_train.takeSample(False, k, seed)
    centers = np.array(centers_list) 
    
    v = np.zeros(k)
    
    # Calculate fraction for pySpark's sample() function
    total_count = rdd_train.count()
    fraction = min(1, float(b)/total_count)
    
    for epoch in range(epochs):
        # Broadcast the centers
        bc_centers = rdd_train.context.broadcast(centers)
        
        # Distributed sampling on workers
        rdd_batch = rdd_train.sample(False, fraction)
        
        # Distributed mapping --> (cluster_idx: (point,1))
        mapped_batch = rdd_batch.map(
            lambda x: (closest_idx(x, bc_centers.value), (x, 1))
        )
        
        # Distributed reduction, collecting only the k aggregates
        reduced_batch = mapped_batch.reduceByKey(
            lambda a, b: (a[0] + b[0], a[1] + b[1])
        ).collect()
        
        # Local update on the Driver 
        for c_idx, (sum_x, count) in reduced_batch:
            v[c_idx] += count               
            eta = count / v[c_idx]          
            batch_mean = sum_x / count      
            centers[c_idx] = (centers[c_idx] * (1.0 - eta)) + (batch_mean * eta)
            
        # Cleans RAM
        bc_centers.destroy()
        
    return centers.tolist()

## Tuning of the `b` Hyperparameter

The Mini-batch $k$-Means algorithm requires to specify the mini-batch size `b` as an input parameter in the `minibatch_kmeans` function, as described in the previous "Spark-parallelized Implementation of Clustering Algorithms" section. In order to choose the optimal value for this parameter, we introduced the `b_search` function, which performs a grid search over a list of candidate values for `b`. Before detailing the `b_search` function, we first to illustrate the implementation of two clustering metrics which were utilized to implement it.

### Clustering Metrics: WCSS and Calinski-Harabasz Index

The first clustering metric we illustrate in this subsection is the Within-Cluster Sum of Squares($WCSS$). It calculates the sum of the squared Euclidean   distances between every data point and the centroid of the cluster it is assigned, measuring the intra-cluster variance:
$$
WCSS = \sum_{k=1}^K\sum_{x_i \in C_k} \|\ \mathbf{x_i} - \mathbf{\mu}_k\|^2,
$$
where
* $\{x_i\}_{C_k}$ are the points assigned to the cluster $C_k$,
* $\mu_k$ is the centroid of the $k$-th cluster,
* $K$ is the total number of clusters.

In the `WCSS` function we introduced a map-reduce approach to compute the $WCSS$ given an RDD and a list of centroids.

```python
def WCSS(rdd_test, centers: list) -> float:
    """Calculates the Within-Cluster Sum of Squares (WCSS) for cost evaluation"""
    centers_np = np.array(centers)
    
    # Broadcasts centers to executors
    bc_centers = rdd_test.context.broadcast(centers_np)
    
    wcss = rdd_test.map(
        lambda x: float(np.sum((x - bc_centers.value[closest_idx(x, bc_centers.value)]) ** 2))
    ).sum()
    
    #Clean executors RAM
    bc_centers.destroy()

    return wcss

The second clustering metric we made use of is the Calinski-Harabasz index ($CH$). This metric computes the ratio of the normalized versions of the Within-Cluster Sum of Squares($WCSS$) and the Between-Cluster Sum of Squares ($BCSS$):
$$
    CH = \frac{BCSS/(K-1)}{WCSS/(n - K)},
$$
where:
* $WCSS$ is the Within-Cluster Sum of Squares,
* $n$ is the number of data points,
* $K$ is the total number of clusters,
* $BCSS$ is the Between-Cluster Sum of Squares, defined as:
    $$
        \sum_{k=1}^K n_k \|\ \mathbf{\mu_k} - \mathbf{\mu}\|^2,
    $$
    where
    - $n_k$ is the number of points in cluster $C_k$,
    - $\mu_k$ is the centroid of cluster $C_k$,
    - $\mu$ is the global centroid, defined as $\frac{1}{n}\sum_{i=1}^n \mathbf{x_i}$.

It is worth noting that $WCSS$ and $BCSS$ are linked by the Law of Total Variance to the Total Sum of Squares ($TSS$):
$$
    TSS \doteq \sum_{i = 1}^n \|\ \mathbf{x_i} - \mathbf{\mu}\|^2 = WCSS + BCSS.
$$
In the `calinski_harabasz` function, we exploit Spark transformations and actions to compute these quantities across the cluster, making use of the aforementioned formulas.

```python
def calinski_harabasz(rdd_test, centers: list, k: int) -> float:
    """Calculates the Calinski-Harabasz Index for best_b identification"""
    N = rdd_test.count()
    
    # Compute gloabl centroid by summing all point vectors and dividing by N
    global_sum = rdd_test.reduce(lambda a, b: a + b)
    global_mean = global_sum / float(N)
    
    # Total Sum of Squares. We use total variance rule to get BCSS later (TSS = BCSS + wcss_score)
    bc_global_mean = rdd_test.context.broadcast(global_mean)

    # Sum of Squares between each point of test set and global center found
    tss = rdd_test.map(
        lambda x: float(np.sum((x - bc_global_mean.value) ** 2))
    ).sum()

    #Clean executors RAM
    bc_global_mean.destroy()
    
    #Evaluates wcss_score using designed functions
    wcss = WCSS(rdd_test, centers)
    
    #BCSS Evaluation
    bcss = tss - wcss
    
    # Avoid zero divisions (just in case)
    if wcss == 0:
        return float('inf')
    
    # Final Index
    ch_score = (bcss / (k - 1)) / (wcss / (N - k))
    
    return ch_score

### Function `b_search` for the Tuning of the `b` Parameter

We can finally detail the `b_search` function. The objective of this function is to find the optimal value for the mini-batch size `b` by performing a grid search. We report below a trimmed version of the full function (which can be consulted in the `functions.py` file), to highlight the core loops and operations involved. Some parts, e.g. the ones regarding metadata logging and the DataFrames operations, have been omitted for brevity.

After train-test split and data caching, the algorithm implements a nested `for` loop. The outer loop runs `num_iter` iterations to ensure statistical robustness. The inner loop iterates on the candidate values of `b`. At each iteration of the inner loop, the model is trained by calling the `minibatch_kmeans` function, and its performance is then evaluated by computing the $CH$ index on the test set, exploiting the `calinski_harabasz` function. Both the score and the execution time are recorded for each configuration.

The optimal value of `b` is then selected by grouping batch sizes within a $2%$ tolerance of the maximum $CH$ score, averaged on the iterations, and picking the one with the smallest execution time.

```python
def b_search(rdd_sample, K, b_list, epochs, num_iter, raw_csv, stats_csv):
    """Grid Search to find best b parameter among list of values"""

    # Tracks total time of the search
    start = time.time()

    # Empty list to store results of every iteration
    results = []

    #pySpark function to split the dataset in training set (80%) and test set (20%)
    rdd_train, rdd_test = rdd_sample.randomSplit([0.8, 0.2], seed=1)

    # Cache the RDDs in memory for the rest of the function
    rdd_train.persist() 
    rdd_test.persist()  

    # Actions to trigger the cache on worker's RAM
    train_size = rdd_train.count() 
    test_size = rdd_test.count()
    
    for run_id in range(num_iter):

        for b in b_list:

            print(f'Testing Mini-Batch Size={b}, iteration:{run_id}')

            #Tracks time of a single iteration
            start_time = time.time()
            centers = minibatch_kmeans(rdd_train, K, b, epochs, seed=run_id)
            exec_time = time.time() - start_time
            
            #Evaluates The evaluation index using the test set and the centers found
            ch_score = calinski_harabasz(rdd_test, centers, K)
            
            # [ ... Data logging ... ]

    # Empty the cache to free up memory on the executors  
    rdd_train.unpersist()
    rdd_test.unpersist()


    # [ ... Pandas aggregation ... ]
    
    # Introduce a tolerance threshold to evaluate only a few CH index vaues in terms of time performance
    best_ch_score = df_stats['mean_ch'].max()
    tolerance = 0.020
    accepted_ch = best_ch_score * (1.0 - tolerance)
    mask_ch = df_stats['mean_ch'] >= accepted_ch

    #Filter the DataFrame and takes only CH values above the threshold
    accepted_batch = df_stats[mask_ch]
    # best_b is the one that minimizes run time among those values
    mask_b = accepted_batch['mean_time'].idxmin()
    best_b = accepted_batch.loc[mask_b, 'batch_size']
    
    # Stop time for the whole grid search
    duration = time.time() - start

    # [... Metadata logging ...]

    return int(best_b), df_stats

## Running of the Algorithms

To run the entire experiment and retrieve the results, the functions `mini_batch_run` and `classic_kmeans_run` were introduced. In this section, we'll limit ourselves to illustrating the workflow of the former, as the two functions are extremely similar one to another. We report below a trimmed version of the full function (which can be consulted in the `functions.py` file), to highlight the core loops and operations involved. Some parts, e.g. the ones regarding metadata logging and the DataFrames operations, have been omitted for brevity.

After the initialization of lists and utility variables and a deterministic train-test split (to ensure a fair comparison across different algorithms), the training and testing RDDs are cached into the Executors' RAM. We encounter the core section of the `mini_batch_run` function, i.e. the `for` loop. During each iteration of this loop, the `minibatch_kmeans` function is called to compute the centroids. The internal seed is varied at each run to ensure the random initialization of the centroids. The execution time is recorded for each training step, and the performance of the model is obtained by computing the $WCSS$ on the test set, exploiting the `WCSS` function. The best model is chosen to be the one minimizing the $WCSS$.

```python


def mini_batch_run(rdd_data, K, best_b, epochs, num_iter, raw_csv, stats_csv):
    """Runs the final mini batch k-means using the discovered optimal parameters"""

    # [ ... Initialization of lists and variables ...]

    #pySpark function to split the dataset in training set (80%) and test set (20%)
    rdd_train, rdd_test = rdd_data.randomSplit([0.8, 0.2],seed=18)

    #Cache the RDDs in memory for the rest of the function
    rdd_train.persist() 
    rdd_test.persist()  

    # Actions to trigger the cache on worker's RAM
    train_size = rdd_train.count() 
    test_size = rdd_test.count()
    
    for run_id in range(num_iter):
        print(f"MiniBatch K-means, {epochs} epochs -- Iteration: {run_id}")
        
        # Time of single iteration
        start_time = time.time()
        centers = minibatch_kmeans(rdd_train, K, best_b, epochs, seed=run_id)
        exec_time = time.time() - start_time
        
        #Evaluates cost function
        wcss = WCSS(rdd_test, centers)

        #Update best values if current wcss is lower than previous best
        if wcss < best_wcss:
            best_wcss = wcss
            best_centers = centers
            best_run_id = run_id
        
        # [ ... Append of results ...]
        
    # Free up cahched memory
    rdd_train.unpersist()
    rdd_test.unpersist()

    # [... Pandas aggregation and logging of metadata ... ]
    
    return df_stats

## Clustering Diagnostic

Once we've retrieved the Champion model running the algorithm with the optimal value for the hyperparameter `b`, the `cluster_diagnostic` function can be used to assess the clustering performance against the ground truth. This function allows to measure the agreement between the discovered clusters and the true labels of the datasets. To do this, the Normalized Mutual Information score (NMI) was used.

We report a streamlined version of the function below, highlighting its core operations.

```python
def cluster_diagnostic(rdd_data, champion_centers, plot_csv):
    """Evaluates Champion Model of a run against the ground truth labels"""

    #Tracks evaluation time
    start = time.time()

    
    # Convert the centers to a numpy array and broadcaste them
    centers_np = np.array(champion_centers)
    bc_centers = rdd_data.context.broadcast(centers_np)

    # Function to create tuple (real_label, predicted_label)
    def predict_label(row):
        # Converts feature in array
        point = np.array(row['features'])

        #Assigns every point (document) of the evaluation dataset to a cluster using the champion centers
        pred = closest_idx(point, bc_centers.value)
    
        return (row['true_labels'], pred)

    #Applies the funtion to the evaluation dataset using map and then collect the results
    labels_rdd = rdd_data.map(predict_label)
    labels_local = labels_rdd.collect()

    # Extracts true and prediceted labels for evaluation
    labels_true = [x[0] for x in labels_local]
    labels_pred = [x[1] for x in labels_local]
    
    print("Evaluating Clustering Performance...")
    # Calcultae NMI using sklearn function
    nmi_score = NMI(labels_true, labels_pred)
    # Lenght of the evaluation dataset to calculate fraction of points to sample
    evaluated_documents = len(labels_true)
    
    #To visualize clusters we just need a few points -> sampling with pyspark sample() function
    fraction = min(1, 5000/evaluated_documents)
    sampled_rows = rdd_data.sample(False, fraction).collect()


    # [ ... Pandas conversion, logging of metadata, plot preparation ... ]
    
    print(f'NMI Score: {nmi_score}')
    
    bc_centers.destroy()
